In [3]:
!pip install langchain langchain-community langchain-experimental

In [1]:
!pip install -U langchain langchain-text-splitters langchain-experimental langchain-community sentence-transformers

In [13]:
# fix_1_chunking.py
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

In [14]:
# Load dataset
df = pd.read_csv("/content/customer_support_tickets.csv")
print(df.columns.tolist())
print(df.head(2))
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
documents = df["ticket_description"].dropna().tolist()

['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']
   Ticket ID  Customer Name              Customer Email  Customer Age  \
0          1  Marisa Obrien  carrollallison@example.com            32   
1          2   Jessica Rios    clarkeashley@example.com            42   

  Customer Gender Product Purchased Date of Purchase      Ticket Type  \
0           Other        GoPro Hero       2021-03-22  Technical issue   
1          Female       LG Smart TV       2021-05-22  Technical issue   

             Ticket Subject  \
0             Product setup   
1  Peripheral compatibility   

                                  Ticket Description  \
0  I'm having an issue with the {product_purchase...   
1  I'm having an issue with t

In [15]:
# Naive chunking
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

In [16]:
# Semantic chunking
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
semantic_splitter = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85  # split when content shifts by > 85th percentile
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
# Compare on first 100 tickets
naive_chunks = []
semantic_chunks = []

for doc in documents[:100]:
    naive_chunks.extend(naive_splitter.split_text(doc))
    semantic_chunks.extend(semantic_splitter.split_text(doc))

print("=" * 50)
print("CHUNKING COMPARISON (first 100 tickets)")
print("=" * 50)
print(f"Naive chunks   : {len(naive_chunks):>5} | avg length: {sum(len(c) for c in naive_chunks)/len(naive_chunks):.0f} chars")
print(f"Semantic chunks: {len(semantic_chunks):>5} | avg length: {sum(len(c) for c in semantic_chunks)/len(semantic_chunks):.0f} chars")

CHUNKING COMPARISON (first 100 tickets)
Naive chunks   :   100 | avg length: 295 chars
Semantic chunks:   200 | avg length: 146 chars


In [18]:
# Show a side-by-side example
sample = documents[0]
print("\n── Sample document ──")
print(sample[:400])

print("\n── Naive split ──")
for i, c in enumerate(naive_splitter.split_text(sample)):
    print(f"  Chunk {i+1}: {c[:120]}...")

print("\n── Semantic split ──")
for i, c in enumerate(semantic_splitter.split_text(sample)):
    print(f"  Chunk {i+1}: {c[:120]}...")


── Sample document ──
I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

── Naive split ──
  Chunk 1: I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that y...

── Semantic split ──
  Chunk 1: I'm having an issue with the {product_purchased}. Please assist. Your billing zip code is: 71701. We appreciate that you...
  Chunk 2: Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue per...
